# 06 — Download and structure the European Commission antitrust registry

## Purpose

This notebook converts the European Commission antitrust case-registry snapshot into
a small set of reproducible tables and downloads the source documents linked to the
Commission decisions used in the dataset.

The notebook is intended to support a **Scientific Data** Data Descriptor. It therefore
keeps the source-specific parsing logic needed for the Commission JSON, but makes the
main data-construction choices explicit and keeps operational download details out of
the publication-facing tables.

The outputs are organised at four levels:

1. **source** — a natural, readable representation of the Commission JSON, with one
   row per case and readable summaries of nested decisions and attachments;
2. **cases** — a concise case-level table;
3. **documents** — one row per Commission decision or case attachment, including
   supporting material outside the study's decision scope;
4. **decisions** — one row per Commission decision included in the study.

A fifth table, **files**, links relevant registry rows to locally retained source files
and SHA-256 checksums.

This notebook does not classify legal citations, reconstruct Court appeal chains, or
harmonise Commission and Court records. Those are downstream stages.


## Source data and frozen snapshot

The source is the European Commission / DG Competition public dataset
**EU Competition: Antitrust and Cartel case publications**.

The local source distribution used here is:

`input/case-data-AT.json`

For the frozen dataset, the registry snapshot is recorded as downloaded on
**30 June 2026**. The notebook also records the SHA-256 hash of the JSON file so the
exact source snapshot can be identified even if the live Commission registry changes.

The raw JSON is never modified. All tables below are derived representations of this
source snapshot.


In [1]:
from __future__ import annotations

import hashlib
import json
import logging
import os
import re
import shutil
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List
from urllib.parse import unquote, urlparse

import pandas as pd
import requests
from tqdm.auto import tqdm


def find_project_root(start: Path | None = None) -> Path:
    override = os.environ.get("ECCJEU_PROJECT_ROOT", "").strip()
    if override:
        root = Path(override).expanduser().resolve()
        if not root.exists():
            raise FileNotFoundError(f"ECCJEU_PROJECT_ROOT does not exist: {root}")
        return root

    start = (start or Path.cwd()).resolve()

    for candidate in [start, *start.parents]:
        if (candidate / ".git").exists():
            return candidate

    for candidate in [start, *start.parents]:
        if (candidate / "input").exists() and (candidate / "output").exists():
            return candidate

    raise FileNotFoundError(
        "Could not locate the repository root. Run the notebook from inside the "
        "repository or set ECCJEU_PROJECT_ROOT."
    )


PROJECT_ROOT = find_project_root()

INPUT_DIR = PROJECT_ROOT / "input"
OUTPUT_DIR = PROJECT_ROOT / "output" / "com_registry"
LOG_DIR = PROJECT_ROOT / "logs" / "com_registry"
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "com_registry"

JSON_PATH = INPUT_DIR / "case-data-AT.json"

SOURCE_CSV = OUTPUT_DIR / "com_registry_source.csv"
CASES_CSV = OUTPUT_DIR / "com_registry_cases.csv"
DOCUMENTS_CSV = OUTPUT_DIR / "com_registry_documents.csv"
DECISIONS_CSV = OUTPUT_DIR / "com_registry_decisions.csv"
FILES_CSV = OUTPUT_DIR / "com_registry_files.csv"
WORKBOOK_XLSX = OUTPUT_DIR / "com_registry.xlsx"

DOWNLOAD_STATE_JSONL = LOG_DIR / "download_state.jsonl"

for directory in [OUTPUT_DIR, LOG_DIR, RAW_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


SOURCE_SNAPSHOT_DATE = "2026-06-30"
SOURCE_DATASET_NAME = "EU Competition: Antitrust and Cartel case publications"
SOURCE_DATASET_URL = (
    "https://data.europa.eu/data/datasets/"
    "18489cb7-bce7-4d44-a138-795b390d2109~~1?locale=en"
)


def repo_relative(path: Path) -> str:
    return path.resolve().relative_to(PROJECT_ROOT.resolve()).as_posix()


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()


if not JSON_PATH.exists():
    raise FileNotFoundError(
        f"Missing source snapshot: {repo_relative(JSON_PATH)}"
    )

SOURCE_JSON_SHA256 = sha256_file(JSON_PATH)

with JSON_PATH.open("r", encoding="utf-8") as handle:
    data = json.load(handle)

print("Project root: .")
print("Source snapshot date:", SOURCE_SNAPSHOT_DATE)
print("Source JSON:", repo_relative(JSON_PATH))
print("Source JSON SHA-256:", SOURCE_JSON_SHA256)
print(f"Cases in source snapshot: {len(data):,}")


Project root: .
Source snapshot date: 2026-06-30
Source JSON: input/case-data-AT.json
Source JSON SHA-256: 6dd7717b98515298095aed0d867c532e1ec31957570fde972a54da765b1c3742
Cases in source snapshot: 751


## 1. Source-specific parsing helpers

The Commission JSON is semi-structured: many fields are lists, JSON-encoded
code/label objects, or nested attachment blocks. Official Journal references also
appear in several forms.

The helper functions below perform only source normalization:

- parse nested code/label structures;
- normalize Commission dates;
- summarize nested decisions and attachments;
- construct EUR-Lex links from Official Journal references;
- extract document links.

References such as `C:2008:021:5` are treated as Official Journal references rather
than being assumed to be CELEX identifiers.


In [2]:
# Utility functions for parsing and summarizing

def parse_json_list(items: List[str]) -> List[Dict[str, Any]]:
    # Parse list of JSON-encoded strings. If an object contains "items", flatten them.
    parsed = []
    for item in items or []:
        try:
            obj = json.loads(item)
            if isinstance(obj, dict) and "items" in obj:
                parsed.extend(obj["items"])
            else:
                parsed.append(obj)
        except Exception:
            parsed.append(item)
    return parsed


def parse_code_label_list(raw_list: List[str]) -> List[str]:
    # Parse list of JSON strings with "code" and "label" into labels.
    labels = []
    for item in raw_list or []:
        try:
            obj = json.loads(item)
            label = obj.get("label")
            if label:
                labels.append(label)
        except Exception:
            if item:
                labels.append(str(item))
    return labels


def first_or_blank(values):
    # Return the first value in a list field, or an empty string.
    if isinstance(values, list):
        return values[0] if values else ""
    return values if values is not None else ""


def join_values(values, sep="; "):
    # Join list values safely for spreadsheet cells.
    if not values:
        return ""
    if not isinstance(values, list):
        return str(values)
    return sep.join(str(v) for v in values if v is not None and str(v) != "")


# --- Date helpers -------------------------------------------------------------
# The Commission API stores dates as ISO strings: yyyy-mm-dd. For manual review
# and Excel output, normalize those values to dd/mm/yyyy everywhere.

def _date_text(value):
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass
    return re.sub(r"\s+", " ", str(value).replace("\u00a0", " " )).strip()


def parse_commission_date(value):
    """Parse a Commission date as yyyy-mm-dd, with dd/mm/yyyy fallback for already-normalized values."""
    text = _date_text(value)
    if not text:
        return pd.NaT

    # Primary expected API format. This avoids month/day ambiguity.
    dt = pd.to_datetime(text, format="%Y-%m-%d", errors="coerce")
    if not pd.isna(dt):
        return dt

    # Fallback for values already normalized by an earlier run or manual edit.
    dt = pd.to_datetime(text, format="%d/%m/%Y", errors="coerce")
    if not pd.isna(dt):
        return dt

    return pd.NaT


def normalize_date(value):
    """Normalize yyyy-mm-dd Commission dates to dd/mm/yyyy. Preserve unparseable values."""
    text = _date_text(value)
    if not text:
        return ""
    dt = parse_commission_date(text)
    if pd.isna(dt):
        return text
    return dt.strftime("%d/%m/%Y")


def normalize_date_list(values):
    if not values:
        return []
    if not isinstance(values, list):
        values = [values]
    return [normalized for normalized in (normalize_date(v) for v in values) if normalized]


def join_dates(values, sep="; "):
    return sep.join(normalize_date_list(values))


def date_sort_series(series):
    return series.map(parse_commission_date)


# --- EUR-Lex URL helpers -----------------------------------------------------
# Commission case JSON references such as "L:2001:166:1" or "C:2008:021:5"
# are Official Journal references: OJ series, year, issue, page. They are not
# always CELEX identifiers.
#
# Preferred generic EUR-Lex OJ URL format used here:
#   https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=uriserv:OJ.L_.2001.166.01.0001.01.ENG&toc=OJ:L:2001:166:TOC
#
# For references that are incomplete or malformed, the helper returns the OJ
# issue table of contents when possible, or an empty string.

def _as_first(value, default=""):
    # Return the first item if value is a list; otherwise return value or default.
    if isinstance(value, list):
        return value[0] if value else default
    return value if value is not None else default


def parse_oj_reference(ref: str) -> Dict[str, Any]:
    # Parse OJ references like C:2008:021:5 or L:2001:166:1.
    # Missing/non-numeric issue or page values are represented as None instead
    # of raising an exception.
    if not ref:
        return {}

    parts = [p.strip() for p in str(ref).strip().split(":")]
    if len(parts) < 3:
        return {"raw": str(ref).strip()}

    series = parts[0].upper() if parts[0] else ""
    year = parts[1] if len(parts) > 1 else ""
    issue_raw = parts[2] if len(parts) > 2 else ""
    page_raw = parts[3] if len(parts) > 3 else ""

    issue = int(issue_raw) if str(issue_raw).isdigit() else None
    page = int(page_raw) if str(page_raw).isdigit() else None

    out = {
        "raw": str(ref).strip(),
        "series": series,
        "year": year,
        "issue": issue,
        "page": page,
        "issue_raw": issue_raw,
        "page_raw": page_raw,
    }

    if series and year and issue is not None:
        out["toc_ref"] = f"OJ:{series}:{year}:{issue:03d}:TOC"

    if series and year and issue is not None and page is not None:
        out["normalized"] = f"{series}:{year}:{issue:03d}:{page}"

    return out


def normalise_oj_reference(ref: str) -> str:
    parsed = parse_oj_reference(ref)
    return parsed.get("normalized") or parsed.get("raw", "")


def eurlex_oj_toc_url(ref: str, lang: str = "EN") -> str:
    parsed = parse_oj_reference(ref)
    toc_ref = parsed.get("toc_ref")
    if not toc_ref:
        return ""
    return f"https://eur-lex.europa.eu/legal-content/{lang}/TXT/?uri={toc_ref}"


def eurlex_oj_uriserv_url(ref: str, lang: str = "EN") -> str:
    parsed = parse_oj_reference(ref)
    series = parsed.get("series")
    year = parsed.get("year")
    issue = parsed.get("issue")
    page = parsed.get("page")

    if not (series and year and issue is not None and page is not None):
        return eurlex_oj_toc_url(ref, lang=lang)

    lang = (lang or "EN").upper()
    lang_suffix_map = {"EN": "ENG", "DE": "DEU", "FR": "FRA", "IT": "ITA", "ES": "SPA", "NL": "NLD"}
    lang_suffix = lang_suffix_map.get(lang, lang)

    uri = f"uriserv:OJ.{series}_.{year}.{issue:03d}.01.{page:04d}.01.{lang_suffix}"
    toc = f"OJ:{series}:{year}:{issue:03d}:TOC"
    return f"https://eur-lex.europa.eu/legal-content/{lang}/TXT/?uri={uri}&toc={toc}"


def eurlex_url_from_oj_item(item: Dict[str, Any], lang: str = "EN") -> str:
    # Return the best available EUR-Lex URL for a case OJ publication item.
    if not isinstance(item, dict):
        return ""

    for key in ("url", "uri", "documentUrl", "eurlexUrl", "officialJournalUrl", "link"):
        value = _as_first(item.get(key), "")
        if isinstance(value, str) and value.startswith("http"):
            return value

    ref = _as_first(item.get("reference"), "")
    return eurlex_oj_uriserv_url(ref, lang=lang)


# --- Item-specific summary helpers ------------------------------------------

def get_oj_label(item):
    # Preserve the most specific OJ label available.
    if not isinstance(item, dict):
        return "OJEU publication"
    return item.get("webDescription", "") or item.get("webDescriptionLabel", "") or "OJEU publication"


def get_view2_oj_label(item):
    # In View 2, keep the broad label "OJEU publication" and append a more
    # specific label when the JSON provides one.
    specific = get_oj_label(item)
    if not specific or specific == "OJEU publication":
        return "OJEU publication"
    return f"OJEU publication: {specific}"


def summarize_press_release_items(raw_list: List[str]) -> List[str]:
    items = parse_json_list(raw_list)
    summaries = []

    for item in items:
        if not isinstance(item, dict) or not item:
            continue

        ref = item.get("reference", "")
        desc = item.get("webDescription", "") or item.get("webDescriptionLabel", "")
        date = item.get("publicationDate", "") or item.get("publishedDate", "")
        url = f"https://ec.europa.eu/commission/presscorner/detail/en/{ref}" if ref else ""

        parts = []
        if ref:
            parts.append(f"Reference: {ref}")
        if desc:
            parts.append(f"Description: {desc}")
        if date:
            parts.append(f"Date: {normalize_date(date)}")
        if url:
            parts.append(f"URL: {url}")

        if parts:
            summaries.append("; ".join(parts))

    return summaries


def summarize_oj_items(raw_list: List[str]) -> List[str]:
    items = parse_json_list(raw_list)
    summaries = []

    for item in items:
        if not isinstance(item, dict) or not item:
            continue

        ref = item.get("reference", "")
        label = get_oj_label(item)
        date = item.get("publishedDate", "")
        url = eurlex_url_from_oj_item(item)

        parts = []
        if ref:
            parts.append(f"Reference: {ref}")
        if label:
            parts.append(f"Label: {label}")
        if date:
            parts.append(f"Date: {normalize_date(date)}")
        if url:
            parts.append(f"URL: {url}")

        if parts:
            summaries.append("; ".join(parts))

    return summaries


def summarize_case_related_information(meta: Dict[str, Any]) -> str:
    # Case-level Official Journal publications and press releases only.
    # Timeline events are intentionally not included here because they generally
    # do not have URLs. OJ and press-release blocks are separated by a blank line
    # for readability in Excel.
    blocks = []

    oj_lines = []
    oj_items = summarize_oj_items(meta.get("caseOfficialJournalPublications", []))
    oj_dates = join_dates(meta.get("caseOfficialJournalPublicationsPublishedDates", []))
    if oj_items or oj_dates:
        oj_lines.append("caseOfficialJournalPublications:")
        for item in oj_items:
            oj_lines.append(f"  - {item}")
        if oj_dates:
            oj_lines.append(f"caseOfficialJournalPublicationsPublishedDates: {oj_dates}")
        blocks.append("\n".join(oj_lines))

    pr_lines = []
    pr_items = summarize_press_release_items(meta.get("casePressReleases", []))
    pr_dates = join_dates(meta.get("casePressReleasesPublicationDates", []))
    if pr_items or pr_dates:
        pr_lines.append("casePressReleases:")
        for item in pr_items:
            pr_lines.append(f"  - {item}")
        if pr_dates:
            pr_lines.append(f"casePressReleasesPublicationDates: {pr_dates}")
        blocks.append("\n".join(pr_lines))

    return "\n\n".join(blocks)

def summarize_attachments(att_list: List[Dict[str, Any]]) -> List[str]:
    # Summarise attachment metadata into readable multi-line field/value blocks.
    # Each returned string represents one attachment item. Metadata type is
    # intentionally omitted because it is not useful for manual review.
    summaries = []

    for att in att_list or []:
        attmeta = att.get("metadata", {}) if isinstance(att, dict) else {}

        cat = parse_code_label_list(attmeta.get("attachmentCategory", []))
        category = cat[0] if cat else "Attachment"

        language = first_or_blank(attmeta.get("attachmentLanguage", [])) or first_or_blank(attmeta.get("language", []))
        document_date = normalize_date(first_or_blank(attmeta.get("attachmentDocumentDate", [])))
        publication_business_date = normalize_date(first_or_blank(attmeta.get("attachmentPublicationBusinessDate", [])))
        publication_description = first_or_blank(attmeta.get("attachmentPublicationDescription", []))
        sent_date = normalize_date(first_or_blank(attmeta.get("attachmentSentDate", [])))
        link = first_or_blank(attmeta.get("attachmentLink", []))
        id_sequence = first_or_blank(attmeta.get("attachmentIdSequence", []))
        metadata_reference = first_or_blank(attmeta.get("metadataReference", []))

        parts = []
        if category:
            parts.append(f"Category: {category}")
        if language:
            parts.append(f"Language: {language}")
        if document_date:
            parts.append(f"Document Date: {document_date}")
        if publication_business_date:
            parts.append(f"Publication Business Date: {publication_business_date}")
        if publication_description:
            parts.append(f"Publication Description: {publication_description}")
        if sent_date:
            parts.append(f"Sent Date: {sent_date}")
        if id_sequence:
            parts.append(f"Attachment ID Sequence: {id_sequence}")
        if metadata_reference:
            parts.append(f"Metadata Reference: {metadata_reference}")
        if link:
            parts.append(f"URL: {link}")

        if parts:
            summaries.append("\n".join(parts))

    return summaries


def attachment_has_url(att: Dict[str, Any]) -> bool:
    if not isinstance(att, dict):
        return False
    attmeta = att.get("metadata", {})
    return bool(first_or_blank(attmeta.get("attachmentLink", [])))

def extract_attachment_links(att_list: List[Dict[str, Any]]) -> List[str]:
    # Extract unique attachmentLink URLs from case or decision attachments.
    links = []
    seen = set()
    for att in att_list or []:
        if not isinstance(att, dict):
            continue
        attmeta = att.get("metadata", {})
        link = first_or_blank(attmeta.get("attachmentLink", []))
        if link and link not in seen:
            seen.add(link)
            links.append(link)
    return links


def extract_oj_links(raw_list: List[str], lang: str = "EN") -> List[str]:
    # Extract unique EUR-Lex / Official Journal links from OJ publication items.
    # This is used separately from attachmentLink because some decisions have
    # no decision attachment, but do have decisionOfficialJournalPublications.
    links = []
    seen = set()
    for item in parse_json_list(raw_list or []):
        if not isinstance(item, dict):
            continue
        link = eurlex_url_from_oj_item(item, lang=lang)
        if link and link not in seen:
            seen.add(link)
            links.append(link)
    return links


## 2. Decision scope

The registry contains many publications besides the Commission decisions used in the
study: press releases, memos, initiation notices, proposed commitments, trustee
material, reports and other supporting documents.

The notebook preserves the **same substantive decision scope as the validated v14
pipeline**. Relevance is a positive inclusion rule using the exact registry labels
observed in the frozen snapshot, plus the existing Article 106 condition for
state-measure decisions.

The previous future-proof regular-expression matching is not used here. For a frozen
dataset, an explicit list of accepted source labels is easier to inspect and reproduce.

Supporting material is not discarded: it remains in the `documents` table with
`relevant = False`. It simply does not enter the decision-level dataset or the
automatic source-document download stage.


In [3]:
RELEVANT_EXACT_ITEM_LABELS = {
    "Prohibition Decision",
    "Prohibition Decision (Art. 7) (Art. 101 & 102 ex 81 & 82)",
    "Commitment Decision",
    "Commitments decision (Art. 9)",
    "Settlement Decision",
    "Exemption without condition (Reg 17/62)",
    "Old milestones - Exemption with condition decision",
    "Negative Clearance Decision (Reg 17/62)",
    "Old milestones - Negative clearance decision",
    "Interim Measures Decision",
    "Interim measures Decision (Art. 8)",
    "Decision imposing fines",
    "Fines Decision (Art. 23)",
    "Periodic Penalty Payment Decision (Art. 24)",
    "Penalty Payments to comply with Commission Decisions",
    "LPP Decision",
    "Amending Decision",
    "Readoption Decision",
    "Re-adoption Decision",
}

ALWAYS_EXCLUDE_ITEM_LABELS = {
    "Rejection of Complaint Decision",
    "Closure of Proceedings",
    "Closure of proceedings",
    "Administrative Closure",
    "Summary Decision",
    "Sector Inquiry - Final Report",
    "Proposed Commitments",
    "Commitments - Final",
    "Cooperation Decision",
    "Trustee Approval Decision",
    "Trustee details",
    "Market test notice (Art. 27(4))",
    "Press Release / Memo",
    "Advisory Committee-Opinion of the Member States",
    "Hearing Officer - Final Report",
    "Initiation of Proceedings",
    "Initiation of proceedings Notice (Art. 11(6))",
    "Informal Guidance Letter",
    "Oral Hearing",
    "Other publication",
    "Provisional non-confidential version of the decision",
    "Report",
    "Response(s) to the Statement of Objections",
}


def is_article_106_legal_basis(legal_basis: Any) -> bool:
    text = str(legal_basis or "").lower()
    return bool(re.search(r"\bart\.?\s*106\b|\barticle\s*106\b", text))


def is_relevant_item_label(item_label: Any, legal_basis: Any = "") -> bool:
    label = str(item_label or "").strip()
    if not label or label in ALWAYS_EXCLUDE_ITEM_LABELS:
        return False

    if label == "State Measure Decision":
        return is_article_106_legal_basis(legal_basis)

    return label in RELEVANT_EXACT_ITEM_LABELS


## 3. Natural source representation

`com_registry_source.csv` is intentionally close to the Commission JSON rather than
being heavily curated.

It contains one row per Commission case. Core metadata are flattened into ordinary
columns, while nested decisions, case attachments and other case-related information
remain as readable multi-line summaries.

This source table is useful for provenance and manual inspection. The smaller `cases`,
`documents` and `decisions` tables created afterwards are the more convenient
analytical representations.


In [4]:
def summarize_decision(decision: Dict[str, Any]) -> str:
    if not decision:
        return ""

    dmeta = decision.get("metadata", {})
    lines = []

    adoption = dmeta.get("decisionAdoptionDate", [])
    if adoption:
        lines.append(f"Adoption date: {normalize_date(adoption[0])}")

    number = dmeta.get("decisionNumber", [])
    if number:
        lines.append(f"Decision number: {number[0]}")

    dtypes = parse_code_label_list(dmeta.get("decisionTypes", []))
    if dtypes:
        lines.append(f"Decision type: {dtypes[0]}")

    oj_list = summarize_oj_items(
        dmeta.get("decisionOfficialJournalPublications", [])
    )
    oj_dates = join_dates(
        dmeta.get("decisionOfficialJournalPublicationsPublishedDates", [])
    )
    if oj_list or oj_dates:
        lines.append("decisionOfficialJournalPublications:")
        for item in oj_list:
            lines.append(f"  - {item}")
        if oj_dates:
            lines.append(
                "decisionOfficialJournalPublicationsPublishedDates: "
                f"{oj_dates}"
            )

    press_releases = summarize_press_release_items(
        dmeta.get("decisionPressReleases", [])
    )
    press_dates = join_dates(
        dmeta.get("decisionPressReleasesPublicationDates", [])
    )
    if press_releases or press_dates:
        lines.append("decisionPressReleases:")
        for item in press_releases:
            lines.append(f"  - {item}")
        if press_dates:
            lines.append(
                f"decisionPressReleasesPublicationDates: {press_dates}"
            )

    attachment_summaries = summarize_attachments(
        decision.get("decisionAttachments", [])
    )
    if attachment_summaries:
        lines.append("decisionAttachments:")
        lines.append("\n\n".join(attachment_summaries))

    return "\n".join(lines)


def build_case_base_row(case_id: str, case: Dict[str, Any]) -> Dict[str, Any]:
    meta = case.get("metadata", {})

    return {
        "case_number": first_or_blank(meta.get("caseNumber", [])) or case_id,
        "case_title": first_or_blank(meta.get("caseTitle", [])),
        "case_instrument": join_values(meta.get("caseInstrument", [])),
        "case_type": join_values(meta.get("caseType", [])),
        "antitrust_cartels": first_or_blank(meta.get("caseCartel", [])),
        "companies": join_values(meta.get("caseCompanies", [])),
        "dg": join_values(meta.get("caseDg", [])),
        "case_number_part": join_values(meta.get("caseNumberPart", [])),
        "last_decision_date": normalize_date(
            first_or_blank(meta.get("caseLastDecisionDate", []))
        ),
        "initiation_of_proceedings": normalize_date(
            first_or_blank(meta.get("caseInitiationDate", []))
        ),
        "economic_activities": join_values(
            parse_code_label_list(meta.get("caseSectors", []))
        ),
        "legal_basis": join_values(
            parse_code_label_list(meta.get("caseLegalBasis", []))
        ),
        "case_language": join_values(meta.get("language", [])),
        "metadata_reference": join_values(meta.get("metadataReference", [])),
    }


def build_source_table(data: Dict[str, Any]) -> pd.DataFrame:
    rows = []

    for case_id, case in data.items():
        meta = case.get("metadata", {})
        row = build_case_base_row(case_id, case)

        decision_summaries = []
        for decision in case.get("decisions", []):
            summary = summarize_decision(decision)
            if summary:
                decision_summaries.append(summary)
        row["decisions"] = "\n\n".join(decision_summaries)

        attachment_summaries = summarize_attachments(
            case.get("caseAttachments", [])
        )
        row["case_attachments"] = "\n\n".join(attachment_summaries)

        other_info = summarize_case_related_information(meta)
        row["other_case_related_information"] = other_info or ""

        rows.append(row)

    source = pd.DataFrame(rows)
    sort_date = date_sort_series(source["initiation_of_proceedings"])

    return (
        source.assign(_sort_date=sort_date)
        .sort_values(
            ["_sort_date", "case_number"],
            na_position="last",
            kind="mergesort",
        )
        .drop(columns="_sort_date")
        .reset_index(drop=True)
    )


source = build_source_table(data)

print(f"Natural source rows: {len(source):,}")
display(source.head())


Natural source rows: 751


,case_number,case_title,case_instrument,case_type,antitrust_cartels,companies,dg,case_number_part,last_decision_date,initiation_of_proceedings,economic_activities,legal_basis,case_language,metadata_reference,decisions,case_attachments,other_case_related_information
0,AT.28841,ABG + AVIA / Gulf + Chevron + Texaco + Mobil +...,Antitrust & Cartels,AtStandardATCCase,Antitrust,Texaco Inc.; Shell Internationale Petroleum Ma...,Competition DG,28841,19/04/1977,04/01/1974,B.06.1 - Extraction of crude petroleum,Art. 102 TFEU,en,AT.28841,Adoption date: 19/04/1977\nDecision number: {C...,,
1,AT.29373,Visa International,Antitrust & Cartels,AtStandardATCCase,Antitrust,Visa International Service Association; Ibanco...,Competition DG,29373,,31/01/1977,K.64.19 - Other monetary intermediation,Art. 101 TFEU,en,AT.29373,,,caseOfficialJournalPublications:\n - Referenc...
2,AT.29629,LAMINES ET ALLIAGES DE ZINC,Antitrust & Cartels,AtStandardATCCase,Cartel,,Competition DG,29629,14/12/1982,09/06/1981,"C.24.43 - Lead, zinc and tin production",,en,AT.29629,Adoption date: 14/12/1982\nDecision number: {3...,,
3,AT.30373,P & I CLUBS,Antitrust & Cartels,AtStandardATCCase,Antitrust,,Competition DG,30373,12/04/1999,18/06/1981,"K.65 - Insurance, reinsurance and pension fund...",Art. 101 TFEU + Art. 102 TFEU,en,AT.30373,Adoption date: 12/04/1999\nDecision number: {5...,,
4,AT.30566,UIP,Antitrust & Cartels,AtStandardATCCase,Antitrust,United International Pictures BV; Paramount Pi...,Competition DG,30566,,11/02/1982,"J.59.13 - Motion picture, video and television...",Art. 101 TFEU,en,AT.30566,,Category: Press Release / Memo\nLanguage: EN\n...,


## 4. Case-level table

`com_registry_cases.csv` contains the concise case-level representation.

The Commission case number is already a meaningful natural key, so no sequential
internal case identifier is added.


In [5]:
CASE_COLUMNS = [
    "case_number",
    "case_title",
    "case_instrument",
    "case_type",
    "antitrust_cartels",
    "companies",
    "dg",
    "case_number_part",
    "last_decision_date",
    "initiation_of_proceedings",
    "economic_activities",
    "legal_basis",
    "case_language",
    "metadata_reference",
]

cases = source[CASE_COLUMNS].copy()

if cases["case_number"].eq("").any():
    raise ValueError("The case table contains an empty case_number.")

if not cases["case_number"].is_unique:
    duplicates = cases.loc[
        cases["case_number"].duplicated(keep=False),
        "case_number",
    ].tolist()
    raise ValueError(
        f"Case numbers are not unique; examples: {duplicates[:10]}"
    )

print(f"Case rows: {len(cases):,}")


Case rows: 751


## 5. Document-level table and identifiers

`com_registry_documents.csv` contains one row per Commission decision object or case
attachment.

The frozen registry already provides a usable unique source identifier for these
rows:

- decision rows use Commission `decisionNumber`;
- case attachments use Commission `metadataReference`.

Those values are therefore used directly as `document_id`. This avoids sequential
identifiers such as `COM1`, `COM2`, ... whose meaning changes when sort order changes.

All document rows remain in the table. `relevant` indicates whether the row enters
the study's Commission-decision universe.

URL provenance is retained in three layers:

- `main_url` — direct item/attachment URL;
- `oj_url` — decision-level Official Journal URL;
- `case_oj_fallback_url` — case-level OJ URL used only when a relevant row has no
  row-level source URL.


In [6]:
def get_case_level_oj_links(
    meta: Dict[str, Any],
    lang: str = "EN",
) -> List[str]:
    return extract_oj_links(
        meta.get("caseOfficialJournalPublications", []),
        lang=lang,
    )


def classify_url_availability(
    main_urls: List[str],
    oj_urls: List[str],
    fallback_urls: List[str],
) -> str:
    if main_urls:
        return "main_url_available"
    if oj_urls:
        return "oj_url_available"
    if fallback_urls:
        return "case_oj_fallback_available"
    return "no_url_available"


def attachment_language(meta: Dict[str, Any]) -> str:
    return (
        first_or_blank(meta.get("attachmentLanguage", []))
        or first_or_blank(meta.get("language", []))
        or ""
    )


def build_documents_table(data: Dict[str, Any]) -> pd.DataFrame:
    rows = []

    for case_id, case in data.items():
        meta = case.get("metadata", {})
        base = build_case_base_row(case_id, case)
        case_oj_links = get_case_level_oj_links(meta)

        for decision in case.get("decisions", []):
            dmeta = decision.get("metadata", {})

            decision_number = first_or_blank(
                dmeta.get("decisionNumber", [])
            )
            adoption_date = normalize_date(
                first_or_blank(dmeta.get("decisionAdoptionDate", []))
            )
            decision_types = parse_code_label_list(
                dmeta.get("decisionTypes", [])
            )
            label = decision_types[0] if decision_types else ""

            main_urls = extract_attachment_links(
                decision.get("decisionAttachments", [])
            )
            oj_urls = extract_oj_links(
                dmeta.get("decisionOfficialJournalPublications", [])
            )

            relevant = is_relevant_item_label(
                label,
                base["legal_basis"],
            )

            fallback_urls = (
                case_oj_links
                if relevant and not main_urls and not oj_urls
                else []
            )

            rows.append({
                "document_id": decision_number,
                "document_id_source": "decisionNumber",
                "case_number": base["case_number"],
                "case_title": base["case_title"],
                "item_type": "Decision",
                "item_date": adoption_date,
                "item_label": label,
                "relevant": relevant,
                "document_language": "",
                "commission_decision_number": decision_number,
                "commission_metadata_reference": "",
                "main_url": join_values(main_urls),
                "oj_url": join_values(oj_urls),
                "case_oj_fallback_url": join_values(fallback_urls),
                "url_availability": classify_url_availability(
                    main_urls,
                    oj_urls,
                    fallback_urls,
                ),
                "item_details": summarize_decision(decision),
            })

        for attachment in case.get("caseAttachments", []):
            if not isinstance(attachment, dict):
                continue

            ameta = attachment.get("metadata", {})
            categories = parse_code_label_list(
                ameta.get("attachmentCategory", [])
            )
            label = categories[0] if categories else "Attachment"

            item_date = normalize_date(
                first_or_blank(ameta.get("attachmentDocumentDate", []))
            )
            main_url = first_or_blank(ameta.get("attachmentLink", []))
            main_urls = [main_url] if main_url else []

            relevant = is_relevant_item_label(
                label,
                base["legal_basis"],
            )

            fallback_urls = (
                case_oj_links
                if relevant and not main_urls
                else []
            )

            metadata_reference = first_or_blank(
                ameta.get("metadataReference", [])
            )

            details = summarize_attachments([attachment])

            rows.append({
                "document_id": metadata_reference,
                "document_id_source": "metadataReference",
                "case_number": base["case_number"],
                "case_title": base["case_title"],
                "item_type": "Case attachment",
                "item_date": item_date,
                "item_label": label,
                "relevant": relevant,
                "document_language": attachment_language(ameta),
                "commission_decision_number": "",
                "commission_metadata_reference": metadata_reference,
                "main_url": main_url,
                "oj_url": "",
                "case_oj_fallback_url": join_values(fallback_urls),
                "url_availability": classify_url_availability(
                    main_urls,
                    [],
                    fallback_urls,
                ),
                "item_details": details[0] if details else "",
            })

    documents = pd.DataFrame(rows)

    sort_date = date_sort_series(documents["item_date"])
    documents = (
        documents.assign(_sort_date=sort_date)
        .sort_values(
            ["case_number", "_sort_date", "item_type", "document_id"],
            na_position="last",
            kind="mergesort",
        )
        .drop(columns="_sort_date")
        .reset_index(drop=True)
    )

    return documents


documents = build_documents_table(data)

if documents["document_id"].fillna("").eq("").any():
    missing = documents.loc[
        documents["document_id"].fillna("").eq(""),
        ["case_number", "item_type", "item_date", "item_label"],
    ]
    raise ValueError(
        "A document row has no Commission source identifier. "
        f"Examples: {missing.head(10).to_dict('records')}"
    )

if not documents["document_id"].is_unique:
    duplicates = documents.loc[
        documents["document_id"].duplicated(keep=False),
        ["document_id", "case_number", "item_type", "item_date"],
    ]
    raise ValueError(
        "Commission document identifiers are not unique. "
        f"Examples: {duplicates.head(10).to_dict('records')}"
    )

unknown_cases = sorted(
    set(documents["case_number"]) - set(cases["case_number"])
)
if unknown_cases:
    raise ValueError(
        f"Documents reference unknown cases: {unknown_cases[:10]}"
    )

print(f"Document rows: {len(documents):,}")
print(
    "Relevant document rows:",
    int(documents["relevant"].fillna(False).astype(bool).sum()),
)
display(documents["document_id_source"].value_counts(dropna=False))


Document rows: 1,189
Relevant document rows: 438


document_id_source
decisionNumber       700
metadataReference    489
Name: count, dtype: int64

## 6. Stable decision identity

The study defines a Commission decision as a unique combination of:

**Commission case number + decision/document date**

This is the same substantive rule used in the validated v14 pipeline, but the
sequential `COMDEC00001`, `COMDEC00002`, ... identifier is removed.

The stable identifier is constructed directly from those defining fields:

`COM_<normalised case number>_<ISO decision date>`

For example:

`AT.38866` + `20/10/2010` → `COM_AT_38866_2010-10-20`

The Commission's own `decisionNumber` is retained separately where available. It is
not used as the universal decision-level key because some historical decisions are
represented only by case attachments.

If several relevant registry rows have the same case and date, they are consolidated
into one decision. If their relevant labels disagree, `decision_label` is set to
`Unclear Decision` and the original labels are retained for manual review.


In [7]:
def clean_text(value: Any) -> str:
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass

    return re.sub(
        r"\s+",
        " ",
        str(value).replace("\u00a0", " "),
    ).strip()


def split_links(value: Any) -> list[str]:
    text = clean_text(value)
    if not text:
        return []

    links = []
    for part in re.split(r"\s*;\s*|\s*\n+\s*", text):
        part = clean_text(part)
        if part.startswith("http"):
            links.append(part)

    return list(dict.fromkeys(links))


def join_unique(values, sep: str = "; ") -> str:
    seen = set()
    output = []

    for value in values:
        text = clean_text(value)
        if not text:
            continue

        for part in re.split(r"\s*;\s*|\n+", text):
            part = part.strip()
            if part and part not in seen:
                seen.add(part)
                output.append(part)

    return sep.join(output)


def date_to_iso(value: Any) -> str:
    text = clean_text(value)
    parsed = pd.to_datetime(
        text,
        format="%d/%m/%Y",
        errors="coerce",
    )
    if pd.isna(parsed):
        raise ValueError(
            f"Could not parse decision date for stable ID: {text!r}"
        )
    return parsed.strftime("%Y-%m-%d")


def normalise_case_for_id(case_number: Any) -> str:
    text = clean_text(case_number).upper()
    text = re.sub(r"[^A-Z0-9]+", "_", text).strip("_")
    if not text:
        raise ValueError("Cannot construct decision_id without a case number.")
    return text


def make_decision_id(case_number: Any, decision_date: Any) -> str:
    return (
        "COM_"
        + normalise_case_for_id(case_number)
        + "_"
        + date_to_iso(decision_date)
    )


def decision_label_from_group(values) -> str:
    labels = [x for x in join_unique(values).split("; ") if x]
    return labels[0] if len(labels) == 1 else "Unclear Decision"


def decision_url_availability(row: pd.Series) -> str:
    if split_links(row.get("main_urls", "")):
        return "main_url_available"
    if split_links(row.get("oj_urls", "")):
        return "oj_url_available"
    if split_links(row.get("case_oj_fallback_urls", "")):
        return "case_oj_fallback_available"
    return "no_url_available"


def build_decisions_table(documents: pd.DataFrame) -> pd.DataFrame:
    relevant = documents.loc[
        documents["relevant"].fillna(False).astype(bool)
    ].copy()

    if relevant["case_number"].fillna("").eq("").any():
        raise ValueError("Relevant document rows contain an empty case_number.")

    if relevant["item_date"].fillna("").eq("").any():
        raise ValueError("Relevant document rows contain an empty item_date.")

    decisions = (
        relevant.groupby(["case_number", "item_date"], dropna=False)
        .agg(
            case_title=("case_title", "first"),
            decision_label=("item_label", decision_label_from_group),
            decision_labels=("item_label", join_unique),
            document_ids=("document_id", join_unique),
            document_count=("document_id", "size"),
            commission_decision_numbers=(
                "commission_decision_number",
                join_unique,
            ),
            item_types=("item_type", join_unique),
            main_urls=("main_url", join_unique),
            oj_urls=("oj_url", join_unique),
            case_oj_fallback_urls=(
                "case_oj_fallback_url",
                join_unique,
            ),
        )
        .reset_index()
        .rename(columns={"item_date": "decision_date"})
    )

    decisions.insert(
        0,
        "decision_id",
        decisions.apply(
            lambda row: make_decision_id(
                row["case_number"],
                row["decision_date"],
            ),
            axis=1,
        ),
    )

    if not decisions["decision_id"].is_unique:
        duplicates = decisions.loc[
            decisions["decision_id"].duplicated(keep=False),
            ["decision_id", "case_number", "decision_date"],
        ]
        raise ValueError(
            "Stable decision IDs are not unique. "
            f"Examples: {duplicates.head(10).to_dict('records')}"
        )

    decisions["is_unclear_decision"] = (
        decisions["decision_label"].eq("Unclear Decision")
    )
    decisions["url_availability"] = decisions.apply(
        decision_url_availability,
        axis=1,
    )

    sort_date = pd.to_datetime(
        decisions["decision_date"],
        format="%d/%m/%Y",
        errors="coerce",
    )

    return (
        decisions.assign(_sort_date=sort_date)
        .sort_values(
            ["_sort_date", "case_number", "decision_id"],
            na_position="last",
            kind="mergesort",
        )
        .drop(columns="_sort_date")
        .reset_index(drop=True)
    )


decisions = build_decisions_table(documents)

decision_lookup = {
    (row.case_number, row.decision_date): row.decision_id
    for row in decisions.itertuples()
}

documents["decision_id"] = documents.apply(
    lambda row: (
        decision_lookup.get(
            (row["case_number"], row["item_date"]),
            "",
        )
        if bool(row["relevant"])
        else ""
    ),
    axis=1,
)

print(f"Decision rows: {len(decisions):,}")
print(
    "Unclear decisions:",
    int(decisions["is_unclear_decision"].sum()),
)
print("Stable decision IDs: unique")


Decision rows: 400
Unclear decisions: 4
Stable decision IDs: unique


## 7. Scope summary

The table below shows every document label observed in the registry view, its row
count, and whether it enters the study's decision scope.

This provides a compact check that the frozen inclusion rule matches the source
snapshot. It is stored in the Excel review workbook but not as an additional
canonical CSV.


In [8]:
scope_summary = (
    documents.groupby(["item_label", "relevant"], dropna=False)
    .size()
    .reset_index(name="rows")
    .sort_values(
        ["relevant", "rows", "item_label"],
        ascending=[False, False, True],
    )
    .reset_index(drop=True)
)

display(scope_summary)


,item_label,relevant,rows
0,Prohibition Decision,True,186
1,Commitment Decision,True,65
2,Prohibition Decision (Art. 7) (Art. 101 & 102 ...,True,52
3,Settlement Decision,True,41
4,Exemption without condition (Reg 17/62),True,21
5,Old milestones - Negative clearance decision,True,17
6,Old milestones - Exemption with condition deci...,True,15
7,State Measure Decision,True,10
8,Amending Decision,True,8
9,Commitments decision (Art. 9),True,6


## 8. File-level manifest

Only relevant document rows enter the automatic source-document download stage.

The same source URL can be associated with more than one registry row. To avoid
duplicating files, `com_registry_files.csv` contains one row per unique source URL and
retains all linked `document_id` and `decision_id` values.

The source role records where the URL came from:

- `main` — direct Commission item/attachment URL;
- `decision_oj` — decision-level Official Journal link;
- `case_oj_fallback` — case-level OJ link used only when no row-level URL exists.

Relevant rows with no available source URL are retained explicitly as
`no_url_available`.


In [9]:
def hash_text(value: Any, n: int = 16) -> str:
    return hashlib.sha256(
        clean_text(value).encode("utf-8")
    ).hexdigest()[:n]


def source_role_pairs(row: pd.Series) -> list[tuple[str, str]]:
    main = split_links(row.get("main_url", ""))
    decision_oj = split_links(row.get("oj_url", ""))
    case_oj = split_links(row.get("case_oj_fallback_url", ""))

    pairs = [("main", url) for url in main]
    pairs.extend(("decision_oj", url) for url in decision_oj)

    if not main and not decision_oj:
        pairs.extend(("case_oj_fallback", url) for url in case_oj)

    return pairs


def build_files_manifest(documents: pd.DataFrame) -> pd.DataFrame:
    relevant = documents.loc[
        documents["relevant"].fillna(False).astype(bool)
    ].copy()

    relations = []
    no_url_rows = []

    for _, row in relevant.iterrows():
        pairs = source_role_pairs(row)

        if not pairs:
            no_url_rows.append({
                "file_id": "COMFILE_NONE_" + hash_text(row["document_id"]),
                "source_url": "",
                "source_roles": "none",
                "document_ids": row["document_id"],
                "decision_ids": row["decision_id"],
                "case_numbers": row["case_number"],
                "item_labels": row["item_label"],
                "status": "no_url_available",
                "selected_url": "",
                "selected_candidate": "",
                "relative_path": "",
                "file_format": "",
                "file_size_bytes": None,
                "sha256": "",
                "http_status": None,
                "error": "No source URL is available in the registry row.",
            })
            continue

        for role, url in pairs:
            relations.append({
                "source_url": url,
                "source_role": role,
                "document_id": row["document_id"],
                "decision_id": row["decision_id"],
                "case_number": row["case_number"],
                "item_label": row["item_label"],
            })

    file_rows = []

    if relations:
        relation_df = pd.DataFrame(relations)

        for source_url, group in relation_df.groupby("source_url", sort=False):
            file_rows.append({
                "file_id": "COMFILE_" + hash_text(source_url),
                "source_url": source_url,
                "source_roles": join_unique(group["source_role"]),
                "document_ids": join_unique(group["document_id"]),
                "decision_ids": join_unique(group["decision_id"]),
                "case_numbers": join_unique(group["case_number"]),
                "item_labels": join_unique(group["item_label"]),
                "status": "pending",
                "selected_url": "",
                "selected_candidate": "",
                "relative_path": "",
                "file_format": "",
                "file_size_bytes": None,
                "sha256": "",
                "http_status": None,
                "error": "",
            })

    columns = [
        "file_id",
        "source_url",
        "source_roles",
        "document_ids",
        "decision_ids",
        "case_numbers",
        "item_labels",
        "status",
        "selected_url",
        "selected_candidate",
        "relative_path",
        "file_format",
        "file_size_bytes",
        "sha256",
        "http_status",
        "error",
    ]

    return pd.DataFrame(
        [*file_rows, *no_url_rows],
        columns=columns,
    )


files = build_files_manifest(documents)

print(f"File-manifest rows: {len(files):,}")
print(
    "Relevant registry rows with no source URL:",
    int(files["status"].eq("no_url_available").sum()),
)


File-manifest rows: 1,213
Relevant registry rows with no source URL: 15


## 9. Resumable downloads

Candidate-level HTTP attempts are operational information and are written only to:

`logs/com_registry/download_state.jsonl`

The notebook shows one `tqdm` progress bar over logical file rows.

For each source URL:

1. try the URL provided by the Commission registry;
2. for EUR-Lex legal-content URLs, try the corresponding HTML representation if
   needed;
3. then try the corresponding PDF representation if needed.

EUR-Lex can return HTTP `202 Accepted` while generating a representation, so a small
bounded retry is retained for that specific behaviour.

Each JSONL event stores the candidate attempts for one logical `file_id`. On a later
run, the latest successful state is reused when the local file still exists. Failed
downloads are retried by default.

The JSONL is an operational checkpoint, not a publication data table.


In [10]:
DOWNLOAD_FILES = True
RETRY_FAILED_DOWNLOADS = True
FORCE_REDOWNLOAD = False

REQUEST_TIMEOUT_SECONDS = 30
REQUEST_SLEEP_SECONDS = 1.0

EURLEX_202_MAX_RETRIES = 3
EURLEX_202_BASE_SLEEP_SECONDS = 5.0
EURLEX_202_MAX_SLEEP_SECONDS = 30.0

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/126.0 Safari/537.36"
    ),
    "Accept": (
        "text/html,application/xhtml+xml,application/xml;q=0.9,"
        "application/pdf;q=0.8,*/*;q=0.7"
    ),
    "Accept-Language": "en-US,en;q=0.9",
}

log_path = LOG_DIR / (
    "download_com_registry_"
    + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    + ".log"
)

logger = logging.getLogger("download_com_registry")
logger.handlers.clear()
logger.setLevel(logging.INFO)

handler = logging.FileHandler(log_path, encoding="utf-8")
handler.setFormatter(
    logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
)
logger.addHandler(handler)
logger.propagate = False


def safe_filename(value: Any, default: str = "file") -> str:
    value = clean_text(value) or default
    value = value.replace("/", "-").replace("\\", "-")
    value = re.sub(r"\s+", "_", value)
    value = re.sub(r"[^A-Za-z0-9._-]+", "_", value)
    return re.sub(r"_+", "_", value).strip("._-") or default


def expand_download_candidates(url: str) -> list[str]:
    url = clean_text(url)
    if not url:
        return []

    if "eur-lex.europa.eu/legal-content/" not in url:
        return [url]

    candidates = [url]
    html_url = None
    pdf_url = None

    if "/TXT/HTML/" in url:
        html_url = url
        pdf_url = url.replace("/TXT/HTML/", "/TXT/PDF/")
    elif "/TXT/PDF/" in url:
        html_url = url.replace("/TXT/PDF/", "/TXT/HTML/")
        pdf_url = url
    elif "/TXT/" in url:
        html_url = url.replace("/TXT/", "/TXT/HTML/")
        pdf_url = url.replace("/TXT/", "/TXT/PDF/")
    elif "/HTML/" in url:
        html_url = url.replace("/HTML/", "/TXT/HTML/")
        pdf_url = url.replace("/HTML/", "/TXT/PDF/")
    elif "/PDF/" in url:
        html_url = url.replace("/PDF/", "/TXT/HTML/")
        pdf_url = url.replace("/PDF/", "/TXT/PDF/")

    for candidate in [html_url, pdf_url]:
        if candidate:
            candidates.append(candidate)

    return list(dict.fromkeys(candidates))


def candidate_kind(source_url: str, candidate_url: str, order: int) -> str:
    if order == 1 or candidate_url == source_url:
        return "original"
    if "/TXT/HTML/" in candidate_url:
        return "eurlex_html_fallback"
    if "/TXT/PDF/" in candidate_url:
        return "eurlex_pdf_fallback"
    return "fallback"


def infer_extension(url: str = "", content_type: str = "") -> str:
    ctype = clean_text(content_type).lower().split(";")[0]

    if ctype == "application/pdf":
        return "pdf"
    if ctype in {"text/html", "application/xhtml+xml"}:
        return "html"
    if ctype == "text/plain":
        return "txt"
    if ctype in {"application/xml", "text/xml"}:
        return "xml"

    suffix = Path(
        unquote(urlparse(clean_text(url)).path)
    ).suffix.lower().lstrip(".")

    if suffix in {"pdf", "html", "htm", "txt", "xml", "doc", "docx"}:
        return "html" if suffix == "htm" else suffix

    return "bin"


def json_safe(value: Any) -> Any:
    if value is None:
        return None
    if isinstance(value, (str, int, bool)):
        return value
    if isinstance(value, float):
        return None if pd.isna(value) else value
    if isinstance(value, Path):
        return value.as_posix()
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]

    try:
        if pd.isna(value):
            return None
    except Exception:
        pass

    return str(value)


def append_download_event(record: dict[str, Any]) -> None:
    with DOWNLOAD_STATE_JSONL.open("a", encoding="utf-8") as handle:
        handle.write(
            json.dumps(
                json_safe(record),
                ensure_ascii=False,
                sort_keys=True,
            )
            + "\n"
        )
        handle.flush()


def load_latest_download_state() -> dict[str, dict[str, Any]]:
    if not DOWNLOAD_STATE_JSONL.exists():
        return {}

    latest = {}

    with DOWNLOAD_STATE_JSONL.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip():
                continue

            try:
                record = json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(
                    f"Malformed download_state.jsonl at line {line_number}: {exc}"
                ) from exc

            file_id = clean_text(record.get("file_id", ""))
            if not file_id:
                raise ValueError(
                    f"download_state.jsonl line {line_number} has no file_id."
                )

            latest[file_id] = record

    return latest


def state_is_reusable(record: dict[str, Any] | None) -> bool:
    if FORCE_REDOWNLOAD or not record:
        return False

    if record.get("status") not in {"downloaded", "already_present"}:
        return False

    relative_path = clean_text(record.get("relative_path", ""))
    if not relative_path:
        return False

    path = PROJECT_ROOT / relative_path
    return path.exists() and path.is_file() and path.stat().st_size > 0


def old_v14_candidates(source_url: str) -> list[Path]:
    old_hash = hash_text(source_url, 10)
    return sorted(
        [
            path
            for path in RAW_DIR.glob(f"*{old_hash}*")
            if path.is_file() and path.stat().st_size > 0
        ],
        key=lambda path: path.stat().st_size,
        reverse=True,
    )


def existing_local_file(file_id: str, source_url: str) -> Path | None:
    stem = safe_filename(file_id)

    for extension in ["pdf", "html", "txt", "xml", "doc", "docx", "bin"]:
        candidate = RAW_DIR / f"{stem}.{extension}"
        if candidate.exists() and candidate.is_file() and candidate.stat().st_size > 0:
            return candidate

    old = old_v14_candidates(source_url)
    return old[0] if old else None


def request_candidate(
    session: requests.Session,
    url: str,
) -> tuple[requests.Response, int]:
    headers = dict(HEADERS)

    if "/TXT/PDF/" in url:
        headers["Accept"] = (
            "application/pdf,application/octet-stream;q=0.9,*/*;q=0.8"
        )

    response = session.get(
        url,
        headers=headers,
        timeout=REQUEST_TIMEOUT_SECONDS,
        allow_redirects=True,
    )

    retries = 0

    while (
        "eur-lex.europa.eu/legal-content/" in url
        and response.status_code == 202
        and retries < EURLEX_202_MAX_RETRIES
    ):
        retries += 1

        retry_after = response.headers.get("Retry-After")
        try:
            sleep_seconds = (
                float(retry_after)
                if retry_after
                else EURLEX_202_BASE_SLEEP_SECONDS * (2 ** (retries - 1))
            )
        except ValueError:
            sleep_seconds = (
                EURLEX_202_BASE_SLEEP_SECONDS * (2 ** (retries - 1))
            )

        time.sleep(
            min(sleep_seconds, EURLEX_202_MAX_SLEEP_SECONDS)
        )

        response = session.get(
            url,
            headers=headers,
            timeout=REQUEST_TIMEOUT_SECONDS,
            allow_redirects=True,
        )

    return response, retries


def looks_like_document(
    content: bytes,
    content_type: str,
    url: str,
) -> bool:
    if not content or len(content) < 200:
        return False

    ctype = clean_text(content_type).lower()

    if "/TXT/PDF/" in url:
        if not content.startswith(b"%PDF") and "pdf" not in ctype:
            return False

    if "html" in ctype:
        sample = content[:20000].decode(
            "utf-8",
            errors="ignore",
        ).lower()

        obvious_error_markers = [
            "request could not be satisfied",
            "page not found",
            "access denied",
            "verify that you're not a robot",
        ]

        if any(marker in sample for marker in obvious_error_markers):
            return False

    return True


def download_file_row(
    session: requests.Session,
    row: dict[str, Any],
) -> dict[str, Any]:
    file_id = row["file_id"]
    source_url = clean_text(row["source_url"])

    if not source_url:
        return {
            **row,
            "status": "no_url_available",
            "updated_at_utc": utc_now_iso(),
            "attempts": [],
        }

    existing = existing_local_file(file_id, source_url)

    if existing is not None and not FORCE_REDOWNLOAD:
        extension = existing.suffix.lower().lstrip(".")
        return {
            **row,
            "status": "already_present",
            "selected_url": source_url,
            "selected_candidate": "existing_local_file",
            "relative_path": repo_relative(existing),
            "file_format": extension,
            "file_size_bytes": existing.stat().st_size,
            "sha256": sha256_file(existing),
            "http_status": None,
            "error": "",
            "updated_at_utc": utc_now_iso(),
            "attempts": [],
        }

    if not DOWNLOAD_FILES:
        return {
            **row,
            "status": "pending",
            "updated_at_utc": utc_now_iso(),
            "attempts": [],
        }

    attempts = []

    for order, candidate_url in enumerate(
        expand_download_candidates(source_url),
        start=1,
    ):
        kind = candidate_kind(source_url, candidate_url, order)

        try:
            response, retries = request_candidate(session, candidate_url)
            content = response.content or b""
            content_type = response.headers.get("Content-Type", "")

            attempt = {
                "candidate": kind,
                "url": candidate_url,
                "http_status": response.status_code,
                "content_type": content_type,
                "bytes": len(content),
                "eurlex_202_retries": retries,
            }

            if (
                response.status_code == 200
                and looks_like_document(content, content_type, candidate_url)
            ):
                extension = infer_extension(candidate_url, content_type)
                output_path = RAW_DIR / f"{safe_filename(file_id)}.{extension}"
                output_path.write_bytes(content)

                attempt["status"] = "downloaded"
                attempts.append(attempt)

                return {
                    **row,
                    "status": "downloaded",
                    "selected_url": candidate_url,
                    "selected_candidate": kind,
                    "relative_path": repo_relative(output_path),
                    "file_format": extension,
                    "file_size_bytes": output_path.stat().st_size,
                    "sha256": sha256_file(output_path),
                    "http_status": response.status_code,
                    "error": "",
                    "updated_at_utc": utc_now_iso(),
                    "attempts": attempts,
                }

            attempt["status"] = "failed"
            attempts.append(attempt)

        except Exception as exc:
            attempts.append({
                "candidate": kind,
                "url": candidate_url,
                "http_status": None,
                "content_type": "",
                "bytes": None,
                "eurlex_202_retries": 0,
                "status": "failed",
                "error": repr(exc),
            })

    error_parts = []
    for attempt in attempts:
        kind = attempt.get("candidate", "candidate")
        status = attempt.get("http_status")
        detail = attempt.get("error", "")

        if status is not None:
            error_parts.append(f"{kind}: HTTP {status}")
        elif detail:
            error_parts.append(f"{kind}: {detail}")
        else:
            error_parts.append(f"{kind}: failed")

    return {
        **row,
        "status": "failed",
        "selected_url": "",
        "selected_candidate": "",
        "relative_path": "",
        "file_format": "",
        "file_size_bytes": None,
        "sha256": "",
        "http_status": attempts[-1].get("http_status") if attempts else None,
        "error": " | ".join(error_parts),
        "updated_at_utc": utc_now_iso(),
        "attempts": attempts,
    }


In [11]:
latest_state = load_latest_download_state()
result_rows = []

session = requests.Session()
session.headers.update(HEADERS)

counts = {
    "ok": 0,
    "failed": 0,
    "no_url": 0,
    "reused": 0,
}

progress = tqdm(
    files.to_dict("records"),
    desc="Downloading COM registry files",
    unit="file",
    dynamic_ncols=True,
)

for row in progress:
    file_id = row["file_id"]
    previous = latest_state.get(file_id)

    if state_is_reusable(previous):
        result = {**row, **previous}
        counts["reused"] += 1

    elif (
        previous
        and previous.get("status") == "failed"
        and not RETRY_FAILED_DOWNLOADS
    ):
        result = {**row, **previous}

    else:
        result = download_file_row(session, row)
        append_download_event(result)
        latest_state[file_id] = result

        if DOWNLOAD_FILES and row.get("source_url"):
            time.sleep(REQUEST_SLEEP_SECONDS)

    result_rows.append({
        column: result.get(column)
        for column in files.columns
    })

    if result.get("status") in {"downloaded", "already_present"}:
        counts["ok"] += 1
    elif result.get("status") == "no_url_available":
        counts["no_url"] += 1
    elif result.get("status") == "failed":
        counts["failed"] += 1

    progress.set_postfix(
        {
            "ok": counts["ok"],
            "fail": counts["failed"],
            "no_url": counts["no_url"],
            "reused": counts["reused"],
        },
        refresh=False,
    )

files = pd.DataFrame(result_rows, columns=files.columns)

print("File status:")
display(files["status"].value_counts(dropna=False))
print(f"Reused successful state rows: {counts['reused']:,}")


File status:


status
downloaded          1150
failed                48
no_url_available      15
Name: count, dtype: int64

Reused successful state rows: 0


## 10. Attach local-file evidence

Downloading source files does not change the identity or relevance of the registry
records. It only adds evidence about which source documents were retained locally.

For documents and consolidated decisions, the notebook records:

- `download_count` — number of successfully retained logical source files;
- `download_paths` — repository-relative paths to those files.

Metadata rows therefore remain visible even when no source file can be downloaded.


In [12]:
SUCCESS_STATUSES = {"downloaded", "already_present"}


def semicolon_values(value: Any) -> list[str]:
    return [
        part.strip()
        for part in str(value or "").split(";")
        if part.strip()
    ]


successful_files = files.loc[
    files["status"].isin(SUCCESS_STATUSES)
].copy()


document_support = []

for document_id in documents["document_id"]:
    related = successful_files.loc[
        successful_files["document_ids"]
        .fillna("")
        .astype(str)
        .map(lambda value: document_id in semicolon_values(value))
    ]

    document_support.append({
        "document_id": document_id,
        "download_count": len(related),
        "download_paths": join_unique(related["relative_path"]),
    })

documents = documents.merge(
    pd.DataFrame(document_support),
    on="document_id",
    how="left",
)


decision_support = []

for decision_id in decisions["decision_id"]:
    related = successful_files.loc[
        successful_files["decision_ids"]
        .fillna("")
        .astype(str)
        .map(lambda value: decision_id in semicolon_values(value))
    ]

    decision_support.append({
        "decision_id": decision_id,
        "download_count": len(related),
        "download_paths": join_unique(related["relative_path"]),
    })

decisions = decisions.merge(
    pd.DataFrame(decision_support),
    on="decision_id",
    how="left",
)

for frame in [documents, decisions]:
    frame["download_count"] = (
        pd.to_numeric(frame["download_count"], errors="coerce")
        .fillna(0)
        .astype(int)
    )
    frame["download_paths"] = frame["download_paths"].fillna("")

print(
    "Relevant decisions without a local source file:",
    int(decisions["download_count"].eq(0).sum()),
)


Relevant decisions without a local source file: 15


## 11. Technical review

The notebook deliberately avoids producing a large validation dataset. Instead it
collects the exceptional cases that are meaningful to inspect before a frozen
Scientific Data release:

- consolidated decisions with conflicting relevant source labels;
- relevant decisions with no retained local source document;
- source URLs for which all download candidates failed.

These rows are placed in the Excel workbook's `review` sheet.

For the final Data Descriptor, these targeted checks should be supplemented by a
stratified manual comparison of ordinary records against the Commission registry.


In [13]:
review_rows = []

for _, row in decisions.loc[
    decisions["is_unclear_decision"]
].iterrows():
    review_rows.append({
        "review_type": "unclear_decision_label",
        "decision_id": row["decision_id"],
        "case_number": row["case_number"],
        "date": row["decision_date"],
        "label_or_url": row["decision_labels"],
        "details": (
            "Multiple relevant registry labels occur for the same "
            "case/date decision."
        ),
    })

for _, row in decisions.loc[
    decisions["download_count"].eq(0)
].iterrows():
    review_rows.append({
        "review_type": "decision_without_local_file",
        "decision_id": row["decision_id"],
        "case_number": row["case_number"],
        "date": row["decision_date"],
        "label_or_url": row["decision_label"],
        "details": row["url_availability"],
    })

for _, row in files.loc[
    files["status"].eq("failed")
].iterrows():
    review_rows.append({
        "review_type": "download_failure",
        "decision_id": row["decision_ids"],
        "case_number": row["case_numbers"],
        "date": "",
        "label_or_url": row["source_url"],
        "details": row["error"],
    })

review = pd.DataFrame(
    review_rows,
    columns=[
        "review_type",
        "decision_id",
        "case_number",
        "date",
        "label_or_url",
        "details",
    ],
)

print(f"Review rows: {len(review):,}")
if len(review):
    display(review["review_type"].value_counts(dropna=False))


Review rows: 67


review_type
download_failure               48
decision_without_local_file    15
unclear_decision_label          4
Name: count, dtype: int64

## 12. Exported data records

The CSV files are the canonical machine-readable outputs:

- `com_registry_source.csv` — natural one-row-per-case representation of the source
  JSON with readable nested summaries;
- `com_registry_cases.csv` — concise case-level metadata;
- `com_registry_documents.csv` — all Commission decision/case-attachment rows with
  source identifiers and relevance flags;
- `com_registry_decisions.csv` — consolidated relevant decisions with stable
  case/date identifiers;
- `com_registry_files.csv` — source-file manifest with repository-relative paths and
  SHA-256 hashes.

`com_registry.xlsx` contains the same main tables for human review, plus `scope`,
`review` and `summary` sheets.

The JSONL under `logs/com_registry` is an operational checkpoint and is not a
canonical scientific data table.


In [14]:
source.to_csv(SOURCE_CSV, index=False, encoding="utf-8")
cases.to_csv(CASES_CSV, index=False, encoding="utf-8")
documents.to_csv(DOCUMENTS_CSV, index=False, encoding="utf-8")
decisions.to_csv(DECISIONS_CSV, index=False, encoding="utf-8")
files.to_csv(FILES_CSV, index=False, encoding="utf-8")


summary = pd.DataFrame(
    [
        ["source", "dataset_name", SOURCE_DATASET_NAME],
        ["source", "dataset_url", SOURCE_DATASET_URL],
        ["source", "snapshot_date", SOURCE_SNAPSHOT_DATE],
        ["source", "source_json_path", repo_relative(JSON_PATH)],
        ["source", "source_json_sha256", SOURCE_JSON_SHA256],
        ["counts", "source_cases", len(source)],
        ["counts", "cases", len(cases)],
        ["counts", "documents", len(documents)],
        [
            "counts",
            "relevant_documents",
            int(documents["relevant"].fillna(False).astype(bool).sum()),
        ],
        ["counts", "decisions", len(decisions)],
        [
            "counts",
            "unclear_decisions",
            int(decisions["is_unclear_decision"].sum()),
        ],
        [
            "counts",
            "decisions_without_local_file",
            int(decisions["download_count"].eq(0).sum()),
        ],
        [
            "counts",
            "failed_source_urls",
            int(files["status"].eq("failed").sum()),
        ],
        [
            "counts",
            "file_rows_without_url",
            int(files["status"].eq("no_url_available").sum()),
        ],
    ],
    columns=["category", "metric", "value"],
)


with pd.ExcelWriter(WORKBOOK_XLSX, engine="openpyxl") as writer:
    source.to_excel(writer, sheet_name="source", index=False)
    cases.to_excel(writer, sheet_name="cases", index=False)
    documents.to_excel(writer, sheet_name="documents", index=False)
    decisions.to_excel(writer, sheet_name="decisions", index=False)
    files.to_excel(writer, sheet_name="files", index=False)
    scope_summary.to_excel(writer, sheet_name="scope", index=False)
    review.to_excel(writer, sheet_name="review", index=False)
    summary.to_excel(writer, sheet_name="summary", index=False)


from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font

workbook = load_workbook(WORKBOOK_XLSX)

for worksheet in workbook.worksheets:
    worksheet.freeze_panes = "A2"

    if worksheet.max_row >= 1 and worksheet.max_column >= 1:
        worksheet.auto_filter.ref = worksheet.dimensions
        for cell in worksheet[1]:
            cell.font = Font(bold=True)

    for column_cells in worksheet.columns:
        letter = column_cells[0].column_letter
        values = [
            str(cell.value or "")
            for cell in column_cells[: min(len(column_cells), 120)]
        ]
        width = min(
            max(max((len(value) for value in values), default=8) + 2, 10),
            50,
        )
        worksheet.column_dimensions[letter].width = width

    for row in worksheet.iter_rows():
        for cell in row:
            if isinstance(cell.value, str) and len(cell.value) > 80:
                cell.alignment = Alignment(
                    vertical="top",
                    wrap_text=True,
                )

workbook.save(WORKBOOK_XLSX)


print("Canonical outputs:")
for path in [
    SOURCE_CSV,
    CASES_CSV,
    DOCUMENTS_CSV,
    DECISIONS_CSV,
    FILES_CSV,
    WORKBOOK_XLSX,
]:
    print("-", repo_relative(path))

print()
print("Operational download state:")
print("-", repo_relative(DOWNLOAD_STATE_JSONL))


Canonical outputs:
- output/com_antitrust/com_registry_source.csv
- output/com_antitrust/com_registry_cases.csv
- output/com_antitrust/com_registry_documents.csv
- output/com_antitrust/com_registry_decisions.csv
- output/com_antitrust/com_registry_files.csv
- output/com_antitrust/com_registry.xlsx

Operational download state:
- logs/com_antitrust/download_state.jsonl


## 13. Technical Validation and Data Records notes

For the frozen Data Descriptor, the most useful technical validation is tied directly
to the construction of the dataset:

- record the exact Commission JSON snapshot date and SHA-256 hash;
- confirm that Commission case numbers are unique;
- confirm that Commission-native document identifiers are present and unique;
- confirm that every relevant document row has a case number and date;
- confirm that the constructed stable `decision_id` values are unique;
- manually review all `Unclear Decision` groups;
- investigate relevant decisions with no retained source file and all failed URLs;
- inspect a stratified sample of ordinary records against the Commission source.

For the **Data Records** section, describe the analytical levels separately:
`source`, `cases`, `documents`, `decisions`, and `files`. The registry source table is
the closest readable representation of the original JSON; the remaining tables are
purposefully simplified views for reuse.

For **Code Availability**, release this notebook together with the repository version
used to produce the deposited dataset. The raw source snapshot, exact source hash,
repository-relative paths and file hashes allow the derived tables and downloaded-file
inventory to be reconstructed.
